In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the dataset 
df = pd.read_csv("student-mat.csv")  # or "student-por.csv"

# Drop non-numeric categorical variables or encode them
categorical_cols = ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 
                    'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 
                    'activities', 'nursery', 'higher', 'internet', 'romantic']

# Encode only the columns that actually exist
encoder = LabelEncoder()
for col in categorical_cols:
    if col in df.columns:
        df[col] = encoder.fit_transform(df[col])
    else:
        print(f"⚠️ Skipping missing column: {col}")

# Check if G3 exists and drop it from features
if 'G3' not in df.columns:
    raise ValueError("Target column 'G3' not found in the dataset!")

# Define features and target
X = df.drop(columns=['G3'])
y = df['G3']

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

# Print metrics
print("\nModel Evaluation:")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R² Score: {r2:.2f}")

# Optional: Feature importance
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': model.coef_})
print("\nTop influential features:")
print(coef_df.sort_values(by='Coefficient', key=abs, ascending=False).head(10))



Model Evaluation:
MAE: 1.50
RMSE: 2.24
R² Score: 0.75

Top influential features:
       Feature  Coefficient
31          G2     0.954569
15   schoolsup     0.750961
18  activities    -0.585128
22    romantic    -0.407915
14    failures    -0.385971
1          sex     0.293869
23      famrel     0.289210
19     nursery    -0.265092
21    internet    -0.234635
30          G1     0.206123


In [4]:
# Use 'G3' instead of 'g3' if that's what your DataFrame has
df['predicted_G3'] = model.predict(df.drop(columns=['G3']))

# Create a result table with actual and predicted grades
result_table = df[['G1', 'G2', 'G3', 'predicted_G3']]

# Round the predictions
result_table['predicted_G3'] = result_table['predicted_G3'].round(2)

# Display the result
print(result_table.head())


<ipython-input-4-6d192caa07fd>:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result_table['predicted_G3'] = result_table['predicted_G3'].round(2)


   G1  G2  G3  predicted_G3
0   5   6   6          5.41
1   5   5   6          4.25
2   7   8  10          6.90
3  15  14  15         13.38
4   6  10  10          9.10
